# 01 · Data audit — DhoNam: Dholuo Speech dataset

**Purpose.** Establish, before any modelling, what the dataset actually contains, what is
broken, and what could leak between training and evaluation. Nothing here trains a model.

**Outputs** (in the notebook output folder, used by notebook 02):

| file | content |
|---|---|
| `manifest.csv` | one row per metadata record, canonical columns, audio/text checks, `keep` flag + `exclude_reason` |
| `audit_summary.json` | every number reported below |
| `audit_report.md` | data-quality report split into **A** facts from the dataset, **B** preprocessing by this project, **C** assumptions |
| `codeswitch_candidates.csv` | tokens flagged as possible English/Swahili, with an empty `verified` column for a Dholuo speaker to fill in |

**Kaggle setup.** Upload the Mozilla Data Collective download as a **private** Kaggle dataset
(NOODL-1.0 forbids re-hosting/re-sharing — do not make it public) and attach it to this
notebook. CPU is enough; no GPU needed.

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

## Parameters
Set `DATA_ROOT` to the attached dataset folder (auto-detected if left `None`). If column auto-detection is wrong, fill `COLUMN_MAP`.

In [ ]:
DATA_ROOT = os.environ.get("DHONAM_ROOT")       # e.g. "/kaggle/input/dhonam-dholuo-speech"
COLUMN_MAP = {}          # e.g. {"audio_path": "file_name", "text_raw": "transcription", "speaker": "speaker_id"}
INCLUDE_FILES = None     # restrict to specific metadata files, e.g. ["train.tsv", "test.tsv"]
QUALITY_DECODE = True    # decode every file for silence/clipping/loudness stats (slow: ~10-30 min for 26k clips)
N_THREADS = 16

# Thresholds for flags/exclusions. These are ENGINEERING ASSUMPTIONS (category C) and are
# written into the report so they can be challenged.
MIN_DURATION_S = 0.3          # shorter = no usable speech -> excluded
MAX_CHARS_PER_SEC = 30.0      # faster than plausible speech -> flagged (audio/transcript mismatch?)
MIN_CHARS_PER_SEC = 1.0       # far slower than plausible -> flagged
LONG_UTTERANCE_S = 20.0       # flagged only (training skips >15 s clips by config)

if DATA_ROOT is None:
    from dholuo_asr.schema import AUDIO_EXTS
    best, best_n = None, 0
    for cand in Path("/kaggle/input").glob("*"):
        n = sum(1 for p in cand.rglob("*") if p.suffix.lower() in AUDIO_EXTS)
        if n > best_n:
            best, best_n = cand, n
    DATA_ROOT = str(best) if best else None
assert DATA_ROOT and Path(DATA_ROOT).exists(), "Set DATA_ROOT to the dataset folder"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

## 1. Inventory
What files exist? Which ones are metadata tables, and what columns do they have?

In [ ]:
from collections import Counter
from dholuo_asr.schema import find_files, read_table
meta_files, audio_files = find_files(DATA_ROOT)
print(f"{len(audio_files):,} audio files;", dict(Counter(p.suffix.lower() for p in audio_files)))
for mf in meta_files:
    try:
        t = read_table(mf)
        print(f"\n{mf.relative_to(DATA_ROOT)}  rows={len(t):,}\n  columns={list(t.columns)}")
    except Exception as e:
        print(f"\n{mf.relative_to(DATA_ROOT)}  UNREADABLE: {e}")
other = [p for p in Path(DATA_ROOT).rglob("*") if p.is_file() and p.suffix.lower() in {".md", ".txt", ".pdf"}]
print("\ndocumentation files:", [str(p.relative_to(DATA_ROOT)) for p in other])

## 2. Canonical manifest
Map provider columns to canonical names and resolve every metadata row to an audio file. **Check the printed mapping** — a wrong mapping (e.g. a prompt id mistaken for a speaker id) would silently break the speaker-independent split.

In [ ]:
from dholuo_asr.schema import build_manifest
man, info = build_manifest(DATA_ROOT, column_map=COLUMN_MAP, include_files=INCLUDE_FILES)
for f in info["files"]:
    print(("USED   " if f["used"] else "skipped"), f["file"], "|", f.get("mapping") or f.get("reason"))
print(f"\nmanifest rows: {len(man):,}")
assert man["speaker"].notna().any(), "No speaker column detected: set COLUMN_MAP['speaker'] (required for leakage-free splits)"
man.head()

## 3. Documented facts vs. observed data
The provider's datasheet (Mozilla Data Collective) states the figures below. Re-deriving them from the files is the first integrity check: disagreement means either a partial download or a wrong column mapping.

In [ ]:
DATASHEET = {   # category A (documented by provider) — source: Mozilla Data Collective datasheet
    "name": "DhoNam: Dholuo Speech dataset",
    "creators": "Dholuo community participants; Maseno Centre for Applied AI (MCAAI), Maseno University; "
                "Centre for Intellectual Property and IT Law (CIPIT), Strathmore University",
    "support": "Mozilla Foundation; GIZ FAIR Forward",
    "platform": "Mozilla Data Collective",
    "license": "NOODL-1.0 (Nwulite Obodo Open Data License); users agree not to re-host or re-share",
    "intended_use": "automatic speech recognition",
    "participants": "adult native speakers (19+) with informed consent",
    "speakers": {"Milambo": 36, "Nyanduat": 23},
    "recordings": {"Milambo": 16697, "Nyanduat": 9394},
}
obs = man.groupby("dialect", dropna=False).agg(recordings=("utt_id", "size"), speakers=("speaker", "nunique"))
obs["documented_recordings"] = obs.index.map(DATASHEET["recordings"])
obs["documented_speakers"] = obs.index.map(DATASHEET["speakers"])
obs

## 4. Missing values and metadata consistency

In [ ]:
canon = ["utt_id", "audio_ref", "text_raw", "speaker", "dialect", "gender", "age", "domain"]
missing = pd.DataFrame({"missing": man[canon].isna().sum() + (man[canon].astype(str).apply(lambda s: s.str.strip()) == "").sum(),
                        "unique_values": man[canon].nunique()})
missing["missing_pct"] = 100 * missing["missing"] / len(man)
display(missing)
for col in ["dialect", "gender", "age", "domain"]:
    if man[col].notna().any():
        print(col, dict(man[col].value_counts(dropna=False).head(15)))
# A speaker should have one dialect/gender/age. More than one = annotation inconsistency (or a reused ID).
inconsistent = {c: int((man.groupby("speaker")[c].nunique() > 1).sum()) for c in ["dialect", "gender", "age"] if man[c].notna().any()}
print("speakers with >1 value:", inconsistent)

## 5. Audio integrity
Header read of every file (sample rate, channels, duration), plus an MD5 of the bytes to find the *same recording* stored under two names.

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor
from dholuo_asr.audio import file_info

def _info(p):
    d = file_info(p)
    if d["readable"]:
        h = hashlib.md5()
        with open(p, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        d["md5"] = h.hexdigest()
    else:
        d["md5"] = None
    return d

paths = [p if isinstance(p, str) and p else None for p in man["audio_path"]]
with ThreadPoolExecutor(N_THREADS) as ex:
    infos = pd.DataFrame(list(ex.map(_info, paths)))
man = pd.concat([man.reset_index(drop=True), infos.rename(columns={"error": "audio_error"})], axis=1)
print("audio status:", dict(man["audio_status"].value_counts()))
print("readable:", int(man["readable"].sum()), "/", len(man))
print("sample rates:", dict(man["sr"].value_counts(dropna=False)))
print("channels:", dict(man["channels"].value_counts(dropna=False)))
print("formats:", dict(man["format"].value_counts(dropna=False).head()))
print(f"total audio: {man['duration_s'].sum()/3600:.2f} h")
print("unreferenced audio files on disk:", len(info["unreferenced_audio"]))
man.loc[~man["readable"], ["utt_id", "audio_ref", "audio_status", "audio_error"]].head(10)

In [ ]:
if QUALITY_DECODE:
    from concurrent.futures import ProcessPoolExecutor
    from dholuo_asr.audio import quality_stats
    ok = man["readable"].to_numpy()
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        q = list(ex.map(quality_stats, [p if k else None for p, k in zip(paths, ok)], chunksize=64))
    man = pd.concat([man, pd.DataFrame(q)], axis=1)
    print(man[["rms_dbfs", "peak", "clip_frac", "lead_sil_s", "trail_sil_s", "speech_frac"]].describe().round(3))
    print("files with >0.1% clipped samples:", int((man["clip_frac"] > 0.001).sum()))
    print("files that are >50% silence:", int((man["speech_frac"] < 0.5).sum()))
else:
    print("quality decode skipped")

## 6. Transcript audit
Character inventory, punctuation/casing variation, apostrophe variants (critical for Dholuo *ng'*), annotation tags, digits, speaking rate (a proxy for audio–transcript misalignment).

In [ ]:
from dholuo_asr.text import normalise, minimal_normalise, has_tags, APOSTROPHE_VARIANTS
man["text_raw"] = man["text_raw"].fillna("").astype(str)
man["text_norm"] = man["text_raw"].map(normalise)
man["n_words"] = man["text_norm"].str.split().str.len().fillna(0).astype(int)
man["n_chars"] = man["text_norm"].str.replace(" ", "").str.len()
man["chars_per_sec"] = man["n_chars"] / man["duration_s"]
man["has_tag"] = man["text_raw"].map(has_tags)
man["has_digit"] = man["text_raw"].str.contains(r"\d")
man["has_upper"] = man["text_raw"].str.contains(r"[A-Z]")
man["has_punct"] = man["text_raw"].str.contains(r"[^\w\s'’‘`´ʼ-]")
man["apostrophe_variant"] = man["text_raw"].map(lambda t: "".join(sorted({c for c in t if c in APOSTROPHE_VARIANTS or c == "'"})))
char_counts = Counter("".join(man["text_raw"]))
print("distinct characters in raw transcripts:", len(char_counts))
print("non-ASCII characters:", {c: n for c, n in char_counts.items() if not c.isascii()})
print("apostrophe forms:", dict(man["apostrophe_variant"].value_counts()))
for c in ["has_tag", "has_digit", "has_upper", "has_punct"]:
    print(f"{c:12s}: {int(man[c].sum()):,} utterances")
print("empty after normalisation:", int((man["text_norm"] == "").sum()))
print("tag examples:", man.loc[man["has_tag"], "text_raw"].head(5).tolist())
print("digit examples:", man.loc[man["has_digit"], "text_raw"].head(5).tolist())
man[["n_words", "n_chars", "duration_s", "chars_per_sec"]].describe().round(2)

## 7. Duplicates and repeated prompts
Read-speech corpora often have many speakers reading the **same prompt**. That is not an error, but if a test prompt also appears in training, a model (and especially a language model) can partly memorise it. This section measures how much that happens.

In [ ]:
dup_rows = man.duplicated(subset=["audio_ref", "text_raw", "speaker"], keep="first")
dup_audio_ref = man.duplicated(subset=["audio_path"], keep="first") & man["audio_path"].notna()
dup_content = man.duplicated(subset=["md5"], keep="first") & man["md5"].notna()
print("exact duplicate metadata rows:", int(dup_rows.sum()))
print("same audio file referenced twice:", int(dup_audio_ref.sum()))
print("identical audio bytes under different names:", int((dup_content & ~dup_audio_ref).sum()))
cross_spk = man[man["md5"].notna()].groupby("md5")["speaker"].nunique()
print("identical audio attributed to >1 speaker:", int((cross_spk > 1).sum()))

prompts = man[man["text_norm"] != ""].groupby("text_norm").agg(n_utts=("utt_id", "size"), n_speakers=("speaker", "nunique"))
shared = prompts["n_speakers"] >= 2
print(f"\nunique prompts: {len(prompts):,}")
print(f"prompts read by >=2 speakers: {int(shared.sum()):,} ({shared.mean():.1%})")
frac_utts_shared = man["text_norm"].map(prompts["n_speakers"]).ge(2).mean()
print(f"utterances whose prompt is read by >=2 speakers: {frac_utts_shared:.1%}")
same_spk_repeat = man.duplicated(subset=["speaker", "text_norm"], keep=False) & (man["text_norm"] != "")
print("utterances where a speaker repeats their own prompt:", int(same_spk_repeat.sum()))
prompts.sort_values("n_speakers", ascending=False).head(10)

## 8. Code-switching candidates
A conservative word-list heuristic (see `src/dholuo_asr/codeswitch.py`). These are **candidates only**: please open `codeswitch_candidates.csv`, mark `verified` = yes/no as a Dholuo speaker, and re-attach it as `codeswitch_candidates_verified.csv` before Experiment E is reported as anything other than *unverified*.

In [ ]:
from dholuo_asr.codeswitch import tag_tokens
tok_tags = man["text_norm"].map(lambda t: tag_tokens(t.split()))
man["cs_candidate"] = tok_tags.map(lambda tags: any(x != "luo?" for x in tags))
cand = Counter((w, tg) for t, tags in zip(man["text_norm"], tok_tags) for w, tg in zip(t.split(), tags) if tg != "luo?")
cs = pd.DataFrame([{"token": w, "guess": g, "count": n} for (w, g), n in cand.most_common()])
if len(cs):
    cs["verified"] = ""
    cs.to_csv(OUT / "codeswitch_candidates.csv", index=False)
print(f"utterances with >=1 candidate token: {int(man['cs_candidate'].sum()):,} ({man['cs_candidate'].mean():.1%})")
cs.head(25)

## 9. Does the provider ship its own split?
If so, check whether it is speaker-independent. If it is not, it cannot be used as the official evaluation.

In [ ]:
split_like = [c for c in man.columns if c.startswith("orig_") and any(k in c for k in ["split", "subset", "partition", "set"])]
src_split = man["source_file"].str.extract(r"(train|dev|valid|validation|val|test)", expand=False)
provider = None
if split_like:
    provider = man[split_like[0]]
elif src_split.notna().any():
    provider = src_split
if provider is not None:
    spk_sets = {s: set(man.loc[provider == s, "speaker"]) for s in provider.dropna().unique()}
    names = list(spk_sets)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            print(f"speaker overlap {names[i]} vs {names[j]}: {len(spk_sets[names[i]] & spk_sets[names[j]])}")
else:
    print("No provider split found — this project defines its own speaker-independent split (notebook 02).")

## 10. Exclusion rules
Only **hard defects** are excluded. Everything else (fast/slow speech, long clips, tags, digits) is flagged and kept so later analysis can study it rather than hide it.

In [ ]:
reason = pd.Series("", index=man.index)
def mark(mask, why):
    global reason
    m = mask & (reason == "")
    reason[m] = why
mark(man["audio_status"] == "missing", "audio_missing")
mark(~man["readable"].astype(bool), "audio_unreadable")
mark(man["duration_s"].fillna(0) < MIN_DURATION_S, "too_short")
mark(man["text_norm"] == "", "empty_transcript")
mark(man["speaker"].isna(), "no_speaker_id")
mark(dup_rows | dup_audio_ref, "duplicate_row")
mark(dup_content, "duplicate_audio_content")
if "decode_ok" in man:
    mark(~man["decode_ok"].astype(bool), "decode_failed")
man["exclude_reason"] = reason
man["keep"] = reason == ""
man["flag_fast_speech"] = man["chars_per_sec"] > MAX_CHARS_PER_SEC
man["flag_slow_speech"] = man["chars_per_sec"] < MIN_CHARS_PER_SEC
man["flag_long"] = man["duration_s"] > LONG_UTTERANCE_S
excl = man["exclude_reason"].replace("", "kept").value_counts()
print(excl.to_string())
print("\nflags among kept rows:", {c: int(man.loc[man.keep, c].sum()) for c in ["flag_fast_speech", "flag_slow_speech", "flag_long", "has_tag", "has_digit", "cs_candidate"]})
print(f"\nkept: {int(man.keep.sum()):,} utterances, {man.loc[man.keep, 'duration_s'].sum()/3600:.2f} h, {man.loc[man.keep, 'speaker'].nunique()} speakers")

## 11. Save manifest, summary and report

In [ ]:
from dholuo_asr.utils import dump_json, env_info
keep_cols = [c for c in man.columns if c not in ("md5",)] + ["md5"]
man[keep_cols].to_csv(OUT / "manifest.csv", index=False)
k = man[man.keep]
summary = {
    "datasheet": DATASHEET, "data_root": DATA_ROOT, "metadata_files": info["files"],
    "rows_total": len(man), "rows_kept": int(man.keep.sum()), "exclusions": excl.to_dict(),
    "hours_total": float(man["duration_s"].sum() / 3600), "hours_kept": float(k["duration_s"].sum() / 3600),
    "speakers_kept": int(k["speaker"].nunique()),
    "by_dialect_kept": k.groupby("dialect", dropna=False).agg(utts=("utt_id", "size"), speakers=("speaker", "nunique"), hours=("duration_s", lambda s: s.sum() / 3600)).round(3).to_dict("index"),
    "gender_speakers": k.groupby("gender", dropna=False)["speaker"].nunique().to_dict(),
    "age_speakers": k.groupby("age", dropna=False)["speaker"].nunique().to_dict(),
    "domain_utts": k["domain"].value_counts(dropna=False).to_dict(),
    "sample_rates": man["sr"].value_counts(dropna=False).to_dict(),
    "unique_prompts": int(len(prompts)), "prompts_shared_by_2plus_speakers": int(shared.sum()),
    "utts_with_shared_prompt_frac": float(frac_utts_shared),
    "speaker_metadata_inconsistencies": inconsistent,
    "transcript_flags": {c: int(man[c].sum()) for c in ["has_tag", "has_digit", "has_upper", "has_punct", "cs_candidate"]},
    "thresholds": {"MIN_DURATION_S": MIN_DURATION_S, "MAX_CHARS_PER_SEC": MAX_CHARS_PER_SEC,
                   "MIN_CHARS_PER_SEC": MIN_CHARS_PER_SEC, "LONG_UTTERANCE_S": LONG_UTTERANCE_S},
    "env": env_info(),
}
dump_json(summary, OUT / "audit_summary.json")

try:
    obs_md = obs.to_markdown()
except ImportError:
    obs_md = "```\n" + obs.to_string() + "\n```"
def fmt(d): return "\n".join(f"| {a} | {b} |" for a, b in d.items())
report = f"""# DhoNam data-quality report (generated by 01_data_audit)

## A1. Facts documented by the provider (datasheet, not re-verified unless stated)
| item | value |
|---|---|
{fmt({k2: v for k2, v in DATASHEET.items()})}

## A2. Facts measured directly from the downloaded files
| item | value |
|---|---|
| metadata rows | {len(man):,} |
| audio files on disk | {len(audio_files):,} |
| unreferenced audio files | {len(info['unreferenced_audio']):,} |
| total audio (all rows) | {summary['hours_total']:.2f} h |
| speakers | {man['speaker'].nunique()} |
| sample rates | {summary['sample_rates']} |
| unique prompts (normalised text) | {len(prompts):,} |
| prompts read by ≥2 speakers | {int(shared.sum()):,} |
| utterances whose prompt is shared across speakers | {frac_utts_shared:.1%} |
| speakers with inconsistent metadata | {inconsistent} |
| transcripts with tags / digits / upper-case / punctuation | {summary['transcript_flags']} |

Observed vs documented per dialect:

{obs_md}

## B. Preprocessing performed by this project
* Exclusions (hard defects only): {excl.to_dict()}
* Kept: {summary['rows_kept']:,} utterances, {summary['hours_kept']:.2f} h, {summary['speakers_kept']} speakers.
* Transcript normalisation (training view `normalised`): NFC, lower-case, apostrophe variants → `'`
  (apostrophe kept: *ng'* is a distinct Dholuo consonant), bracketed tags removed, punctuation and
  hyphens removed, whitespace collapsed. Evaluation applies the same normaliser to reference and hypothesis.
* Audio: decoded, down-mixed to mono and resampled to 16 kHz at load time; no denoising, trimming or gain change.

## C. Assumptions made by the engineering team
* Column mapping: {[f['mapping'] for f in info['files'] if f['used']]}
* A speaker's identity is the provider's speaker id; ids are assumed not to be reused across people.
* Thresholds: {summary['thresholds']} (flags, except MIN_DURATION_S which excludes).
* Code-switching flags come from a word-list heuristic and are unverified until checked by a Dholuo speaker.
"""
(OUT / "audit_report.md").write_text(report)
print(report[:3000])